# CNN — Conv2D con 32 kernels SIN entrenamiento

Este Colab muestra qué ocurre cuando una imagen de **28×28×1** pasa por una capa:

```python
tf.keras.layers.Conv2D(32, (3,3), input_shape=(28,28,1))
```

**No se entrena la red.** Los 32 kernels permanecen con sus valores iniciales aleatorios.

Se visualizan:
1. La imagen de entrada.
2. Los 32 kernels iniciales.
3. Los 32 mapas de características producidos por esos kernels.
4. Las dimensiones de entrada, kernels y salida.

> Importante: sin entrenamiento, los mapas de características **no representan características aprendidas**. Son simplemente la respuesta de los filtros iniciales aleatorios.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf

# Semilla para que los resultados sean reproducibles
np.random.seed(42)
tf.random.set_seed(42)

print('TensorFlow:', tf.__version__)

## 1. Crear la capa Conv2D

La capa tiene **32 filtros**, cada uno de tamaño **3×3**.

Como la entrada es una imagen en escala de grises, cada kernel tiene forma 3×3×1.

In [ ]:
conv = tf.keras.layers.Conv2D(
    filters=32,
    kernel_size=(3, 3),
    padding='valid',
    input_shape=(28, 28, 1),
    name='conv2d_32'
)

conv.build((None, 28, 28, 1))

kernels, biases = conv.get_weights()

print('Entrada:              28 × 28 × 1')
print('Número de kernels:    32')
print('Tamaño de cada kernel: 3 × 3 × 1')
print('Forma de kernels:', kernels.shape)
print('Forma de biases: ', biases.shape)

## 2. Imagen de entrada

Para no entrenar nada, usamos directamente una imagen de ejemplo de MNIST. Solo se descarga la base para obtener una imagen; **no se realiza ningún entrenamiento**.

In [ ]:
(x_train, y_train), (x_test, y_test) = tf.keras.datasets.mnist.load_data()

# Tomamos una sola imagen
indice = 0
imagen = x_test[indice].astype('float32') / 255.0

# Convertir 28×28 -> 1×28×28×1
imagen_tensor = imagen[np.newaxis, :, :, np.newaxis]

print('Dígito:', y_test[indice])
print('Forma de imagen:', imagen.shape)
print('Forma para Conv2D:', imagen_tensor.shape)

plt.figure(figsize=(4,4))
plt.imshow(imagen, cmap='gray')
plt.title(f'Imagen de entrada — dígito {y_test[indice]}')
plt.axis('off')
plt.show()

## 3. Visualizar los 32 kernels iniciales

Estos filtros todavía **no han aprendido nada**.

In [ ]:
fig, axes = plt.subplots(4, 8, figsize=(14, 7))

for i, ax in enumerate(axes.flat):
    kernel = kernels[:, :, 0, i]
    im = ax.imshow(kernel, cmap='gray', interpolation='nearest')
    ax.set_title(f'Kernel {i+1}')
    ax.set_xticks([])
    ax.set_yticks([])

fig.suptitle('32 kernels iniciales — SIN entrenamiento', fontsize=18, fontweight='bold')
plt.tight_layout()
plt.show()

## 4. Aplicar los 32 kernels a la imagen

Ahora hacemos únicamente la operación convolucional:

**28×28×1 → Conv2D(32, 3×3) → 26×26×32**

No hay `fit()`, no hay backpropagation y los kernels no cambian.

In [ ]:
salida = conv(imagen_tensor)
salida_np = salida.numpy()

print('Forma de salida:', salida_np.shape)
print('Resultado esperado: (1, 26, 26, 32)')


## 5. Visualizar los 32 mapas de características

Cada mapa corresponde a la respuesta de **un kernel** sobre la misma imagen.

In [ ]:
fig, axes = plt.subplots(4, 8, figsize=(14, 8))

for i, ax in enumerate(axes.flat):
    feature_map = salida_np[0, :, :, i]
    ax.imshow(feature_map, cmap='gray')
    ax.set_title(f'Salida K{i+1}')
    ax.axis('off')

fig.suptitle('32 mapas de características — kernels SIN entrenamiento',
             fontsize=18, fontweight='bold')
plt.tight_layout()
plt.show()

## 6. Ver un kernel y su salida con más detalle

Cambia `k` para observar otro filtro.

In [ ]:
k = 15  # Kernel que queremos inspeccionar (0 a 31)

kernel = kernels[:, :, 0, k]
feature_map = salida_np[0, :, :, k]

fig, ax = plt.subplots(1, 2, figsize=(10, 4))

ax[0].imshow(kernel, cmap='gray', interpolation='nearest')
ax[0].set_title(f'Kernel {k+1} — 3×3')
ax[0].axis('off')

ax[1].imshow(feature_map, cmap='gray')
ax[1].set_title(f'Mapa producido — 26×26')
ax[1].axis('off')

plt.tight_layout()
plt.show()

## 7. Resumen de dimensiones

| Elemento | Dimensión |
|---|---:|
| Imagen de entrada | 28 × 28 × 1 |
| Número de kernels | 32 |
| Tamaño de cada kernel | 3 × 3 × 1 |
| Bias | 32 |
| Padding | valid |
| Stride | 1 |
| Cada mapa de salida | 26 × 26 |
| Salida completa | 26 × 26 × 32 |

La relación es:

**28×28×1 → 32 convoluciones 3×3 → 26×26×32**